In [ ]:
from pathlib import Path

# Tìm thư mục gốc TiKi-Analysic dù chạy notebook từ thư mục code/ hay root project.
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "01_craw_data").is_dir():
    PROJECT_DIR = CURRENT_DIR
elif CURRENT_DIR.name == "code" and CURRENT_DIR.parent.name == "01_craw_data":
    PROJECT_DIR = CURRENT_DIR.parent.parent
else:
    PROJECT_DIR = next((p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "01_craw_data").is_dir()), CURRENT_DIR)

INPUT_DIR = PROJECT_DIR / "01_craw_data" / "categories"
DATA_DIR = PROJECT_DIR / "01_craw_data" / "products"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CATEGORIES_FILE = INPUT_DIR / "categories.csv"
SUB_CATEGORY_FILE = INPUT_DIR / "sub_category.csv"
PRODUCT_ID_FILE = INPUT_DIR / "product_id.csv"


# 1. CATEGORY

In [ ]:
import requests
import pandas as pd
from urllib.parse import urlparse

url = "https://api.tiki.vn/raiden/v2/menu-config"
params = {"platform": "desktop", "30122025": 1}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://tiki.vn/"
}

response = requests.get(url, params=params, headers=headers, timeout=30)

if response.status_code == 200:
    items = response.json().get("menu_block", {}).get("items", [])
    categories = []

    for item in items:
        path = urlparse(item.get("link", "")).path.rstrip("/").split("/")
        cid = path[-1][1:] if path else ""

        if cid.isdigit():
            categories.append({
                "category_id": int(cid),
                "name": item.get("text"),
                "urlKey": path[-2],
                "src": f"c{cid}"
            })

    df = pd.DataFrame(categories).drop_duplicates("category_id")
    df.to_csv(CATEGORIES_FILE, index=False, encoding="utf-8-sig")

    print(f"Đã lưu {len(df)} danh mục vào categories.csv")
else:
    print("Request thất bại:", response.status_code)

# 2. Sub_Categories

In [ ]:
import requests
import pandas as pd
import time

# Cấu hình
START_ROW = 1
END_ROW = None
SAVE_EVERY = 100

df = pd.read_csv(CATEGORIES_FILE)
id_col = "category_id" if "category_id" in df.columns else "id"
df = df.drop_duplicates(id_col).iloc[START_ROW - 1:END_ROW]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://tiki.vn/"
})

total_count = 0

def save_data(rows):
    if rows:
        old = pd.read_csv(SUB_CATEGORY_FILE) if SUB_CATEGORY_FILE.exists() else pd.DataFrame()
        pd.concat([old, pd.DataFrame(rows)], ignore_index=True).drop_duplicates() \
            .to_csv(SUB_CATEGORY_FILE, index=False, encoding="utf-8-sig")
        rows.clear()

def get_children(parent_id):
    try:
        r = session.get(
            "https://tiki.vn/api/v2/categories",
            params={"include": "children", "parent_id": parent_id},
            timeout=20
        )
        r.raise_for_status()
        data = r.json()
        return data.get("data", data.get("children", []))
    except Exception as e:
        print(f"Lỗi {parent_id}: {e}")
        return None

def dfs(root_id, root_name, parent_id, path, rows):
    global total_count

    children = get_children(parent_id)
    if children is None:
        return False

    if not children:
        if path:
            row = {"category_id": root_id, "category_name": root_name}
            for i, item in enumerate(path, 1):
                row[f"sub_category_{i}_id"] = item["id"]
                row[f"sub_category_{i}"] = item["name"]
            rows.append(row)
            total_count += 1

            if len(rows) >= SAVE_EVERY:
                save_data(rows)
        return True

    success = True
    for child in children:
        if child.get("id") is not None:
            if not dfs(
                root_id, root_name, child["id"],
                path + [{"id": child["id"], "name": child.get("name")}],
                rows
            ):
                success = False
            time.sleep(0.3)

    return success

rows = []

for _, root in df.iterrows():
    start_count = total_count
    success = dfs(int(root[id_col]), root["name"], int(root[id_col]), [], rows)

    if success and total_count > start_count:
        print(f"Craw: {root['name']} - {total_count - start_count} sub_categories")
    else:
        print(f"Chưa lấy đủ dữ liệu: {root['name']}")

save_data(rows)
print("Hoàn tất crawl danh mục!")

# 3. PRODUCT_ID

In [ ]:
import requests
import time
import random
import pandas as pd

# Cấu hình
START_ROW = 1
END_ROW = None
SAVE_EVERY = 100

categories = pd.read_csv(CATEGORIES_FILE).drop_duplicates("category_id")
categories = categories.iloc[START_ROW - 1:END_ROW]

url = "https://tiki.vn/api/personalish/v1/blocks/listings"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*"
}

def save_data(rows):
    if rows:
        pd.DataFrame(rows).drop_duplicates().to_csv(
            PRODUCT_ID_FILE, mode="a", index=False,
            header=not PRODUCT_ID_FILE.exists(), encoding="utf-8-sig"
        )
        rows.clear()

with requests.Session() as session:
    session.headers.update(headers)

    for _, cat in categories.iterrows():
        category_id = int(cat["category_id"])
        params = {
            "category": category_id,
            "src": cat["src"],
            "urlKey": cat["urlKey"]
        }
        product_data = []
        count = 0

        for page in range(1, 101):
            params["page"] = page

            try:
                response = session.get(url, params=params, timeout=30)
                response.raise_for_status()
                records = response.json().get("data") or []
                if not records:
                    break

                for r in records:
                    if r.get("id") is not None:
                        product_data.append({
                            "product_id": r["id"],
                            "seller_product_id": r.get("seller_product_id"),
                            "category_id": category_id
                        })
                        count += 1

                        if len(product_data) >= SAVE_EVERY:
                            save_data(product_data)

            except (requests.RequestException, ValueError):
                break

            time.sleep(random.uniform(3, 6))

        save_data(product_data)
        print(f"Crawl: {cat['name']} - {count} product_id")
        time.sleep(random.uniform(3, 6))
        
if PRODUCT_ID_FILE.exists():
    df = pd.read_csv(PRODUCT_ID_FILE)

    df = df.drop_duplicates(
        subset=["product_id", "seller_product_id", "category_id"]
    )

    df.to_csv(PRODUCT_ID_FILE, index=False, encoding="utf-8-sig")

print("\nHoàn tất crawl tất cả danh mục!")

# 4. First Craw

In [ ]:
import requests
import pandas as pd
import time
import random
import json
from datetime import datetime, timedelta
from pathlib import Path

# =========================
# 1. CẤU HÌNH
# =========================

INPUT_FILE = str(PRODUCT_ID_FILE)

# Số dòng tính từ 1, không tính dòng tiêu đề
START_ROW = 1
END_ROW = None      # None = chạy đến dòng cuối

SAVE_EVERY = 100      # Lưu sau mỗi 10 sản phẩm
SLEEP_MIN = 3
SLEEP_MAX = 5

OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "product": OUTPUT_DIR / "product.csv",
    "category": OUTPUT_DIR / "product_category.csv",
    "seller": OUTPUT_DIR / "product_seller.csv",
    "history": OUTPUT_DIR / "product_price_history.csv"
}

# =========================
# 2. SCHEMA 4 BẢNG
# =========================

PRODUCT_COLS = [
    "product_id", "master_id", "sku", "product_name",
    "product_type", "url_key", "day_ago_created",
    "favourite_count", "brand_id", "brand_name",
    "created_date"
]

CATEGORY_COLS = [
    "product_id", "category_id", "category_name",
    "category_level", "category_url", "is_leaf",
    "is_primary"
]

SELLER_COLS = [
    "product_id", "seller_product_id", "seller_id",
    "seller_name", "seller_link", "store_id",
    "seller_price", "is_current_seller", "is_best_store"
]

HISTORY_COLS = [
    "product_id", "seller_product_id", "seller_id",
    "crawl_time", "crawl_date", "price", "list_price",
    "discount", "discount_rate", "rating_average",
    "review_count", "quantity_sold",
    "all_time_quantity_sold", "favourite_count",
    "inventory_status", "stock_qty"
]

# Khóa dùng để chống trùng
KEYS = {
    "product": ["product_id"],
    "category": ["product_id", "category_id"],
    "seller": [
        "product_id", "seller_product_id", "seller_id"
    ],
    "history": [
        "product_id", "seller_product_id",
        "seller_id", "crawl_time"
    ]
}

COLUMNS = {
    "product": PRODUCT_COLS,
    "category": CATEGORY_COLS,
    "seller": SELLER_COLS,
    "history": HISTORY_COLS
}

# =========================
# 3. HÀM XỬ LÝ DỮ LIỆU
# =========================

def get_number(value):
    """Lấy giá trị số, giữ trống nếu không có dữ liệu."""
    if value is None or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return value

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return None
        try:
            return float(value.replace(",", ""))
        except ValueError:
            return None

    return None


def get_sold_value(data):
    """Lấy quantity_sold.value nếu API trả về object."""
    value = data.get("quantity_sold")

    if isinstance(value, dict):
        return get_number(value.get("value"))

    return get_number(value)


def get_bool(value):
    """Chuẩn hóa boolean, không biến dữ liệu thiếu thành False."""
    if value is None or value == "":
        return None

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        return bool(value)

    if isinstance(value, str):
        value = value.strip().lower()
        if value in ("true", "1", "yes"):
            return True
        if value in ("false", "0", "no"):
            return False

    return None


def get_category_id(category):
    """Hỗ trợ JSON dùng id hoặc category_id."""
    value = category.get("category_id")
    if value is None:
        value = category.get("id")
    return str(value) if value is not None else ""


def get_brand(data):
    brand = data.get("brand") or {}

    if not isinstance(brand, dict):
        return None, None

    return brand.get("id"), brand.get("name")


# =========================
# 4. LƯU CSV VÀ CHỐNG TRÙNG
# =========================

def save_table(table_name, new_rows):
    """
    Gộp dữ liệu mới với file cũ.
    Các bảng thông tin hiện tại cập nhật bản ghi trùng khóa.
    Bảng history chỉ loại bản ghi trùng cùng crawl_time.
    """
    if not new_rows:
        return

    path = FILES[table_name]
    columns = COLUMNS[table_name]
    key_cols = KEYS[table_name]

    new_df = pd.DataFrame(new_rows, columns=columns)
    new_df = new_df.fillna("").astype(str)

    if path.exists() and path.stat().st_size > 0:
        old_df = pd.read_csv(path, dtype=str).fillna("")

        # Bổ sung cột nếu file cũ thiếu cột
        for col in columns:
            if col not in old_df.columns:
                old_df[col] = ""

        old_df = old_df[columns].astype(str)
        combined = pd.concat(
            [old_df, new_df], ignore_index=True
        )
    else:
        combined = new_df

    combined = combined.drop_duplicates(
        subset=key_cols,
        keep="last"
    )

    combined.to_csv(path, index=False, encoding="utf-8-sig")


def save_all(rows):
    """Lưu đồng thời 4 bảng."""
    for name in COLUMNS:
        save_table(name, rows[name])


# =========================
# 5. PARSE PRODUCT
# =========================

def parse_product(data, crawl_date):
    brand_id, brand_name = get_brand(data)

    day_ago = get_number(data.get("day_ago_created"))

    # Ngày tạo chỉ là giá trị ước tính
    created_date = ""
    if day_ago is not None and day_ago >= 0:
        created_date = (
            crawl_date - timedelta(days=int(day_ago))
        ).isoformat()

    return {
        "product_id": data.get("id"),
        "master_id": data.get("master_id"),
        "sku": data.get("sku"),
        "product_name": data.get("name") or data.get("meta_title"),
        "product_type": data.get("product_type"),
        "url_key": data.get("url_key"),
        "day_ago_created": day_ago,
        "favourite_count": data.get("favourite_count"),
        "brand_id": brand_id,
        "brand_name": brand_name,
        "created_date": created_date
    }


# =========================
# 6. PARSE CATEGORY
# =========================

def parse_categories(data, product_id):
    rows = []

    breadcrumbs = data.get("breadcrumbs") or []
    if not isinstance(breadcrumbs, list):
        breadcrumbs = []

    category_info = data.get("categories") or {}
    if not isinstance(category_info, dict):
        category_info = {}

    primary_id = category_info.get("id")
    if primary_id is not None:
        primary_id = str(primary_id)

    # Lấy breadcrumb hợp lệ, bỏ category_id = 0
    valid_breadcrumbs = []
    seen_ids = set()

    for item in breadcrumbs:
        if not isinstance(item, dict):
            continue

        cid = get_category_id(item)

        if not cid or cid == "0" or cid in seen_ids:
            continue

        seen_ids.add(cid)
        valid_breadcrumbs.append(item)

    # Nếu danh mục chính không có trong breadcrumbs thì bổ sung
    if primary_id and primary_id != "0" and primary_id not in seen_ids:
        valid_breadcrumbs.append({
            "id": primary_id,
            "name": category_info.get("name"),
            "url": category_info.get("url")
        })

    for level, item in enumerate(valid_breadcrumbs, start=1):
        cid = get_category_id(item)

        # Chỉ biết is_leaf nếu item khớp categories.id
        is_leaf = None
        if cid == primary_id:
            is_leaf = get_bool(category_info.get("is_leaf"))

        rows.append({
            "product_id": product_id,
            "category_id": cid,
            "category_name": item.get("name"),
            "category_level": level,
            "category_url": item.get("url"),
            "is_leaf": is_leaf,
            "is_primary": cid == primary_id if primary_id else None
        })

    return rows


# =========================
# 7. PARSE SELLERS
# =========================

def parse_sellers(data, requested_spid):
    sellers = []

    current = data.get("current_seller")

    if isinstance(current, dict) and current:
        sellers.append((current, True))

    other_sellers = data.get("other_sellers") or []

    if isinstance(other_sellers, list):
        for seller in other_sellers:
            if isinstance(seller, dict):
                sellers.append((seller, False))

    result = []
    seen = set()

    for seller, is_current in sellers:
        # seller.product_id là ID sản phẩm của người bán
        seller_product_id = seller.get("product_id")

        if seller_product_id is None and is_current:
            seller_product_id = requested_spid

        seller_id = seller.get("id")
        if seller_id is None:
            seller_id = seller.get("seller_id")

        seller_product_id = (
            str(seller_product_id)
            if seller_product_id is not None else ""
        )
        seller_id = str(seller_id) if seller_id is not None else ""

        key = (seller_product_id, seller_id)

        if key in seen:
            continue
        seen.add(key)

        result.append({
            "product_id": data.get("id"),
            "seller_product_id": seller_product_id,
            "seller_id": seller_id,
            "seller_name": seller.get("name"),
            "seller_link": seller.get("link"),
            "store_id": seller.get("store_id"),
            "seller_price": get_number(seller.get("price")),
            "is_current_seller": is_current,
            "is_best_store": get_bool(seller.get("is_best_store"))
        })

    # Nếu API không trả seller, vẫn giữ được snapshot sản phẩm.
    # Seller ID không xác định sẽ để trống, không tự bịa seller_id.
    if not result:
        result.append({
            "product_id": data.get("id"),
            "seller_product_id": str(requested_spid),
            "seller_id": "",
            "seller_name": None,
            "seller_link": None,
            "store_id": None,
            "seller_price": get_number(data.get("price")),
            "is_current_seller": True,
            "is_best_store": None
        })

    return result


# =========================
# 8. PARSE PRICE HISTORY
# =========================

def parse_history(data, seller_rows, crawl_time, crawl_date):
    sold = get_sold_value(data)

    stock_item = data.get("stock_item") or {}
    if not isinstance(stock_item, dict):
        stock_item = {}

    rows = []

    for seller in seller_rows:
        # Giá của từng người bán; chỉ dùng giá gốc
        # khi không có giá seller và đó là dòng đại diện.
        price = seller.get("seller_price")

        if price is None and seller.get("is_current_seller"):
            price = get_number(data.get("price"))

        rows.append({
            "product_id": data.get("id"),
            "seller_product_id": seller.get("seller_product_id"),
            "seller_id": seller.get("seller_id"),
            "crawl_time": crawl_time,
            "crawl_date": crawl_date.isoformat(),
            "price": price,
            "list_price": get_number(data.get("list_price")),
            "discount": get_number(data.get("discount")),
            "discount_rate": get_number(data.get("discount_rate")),
            "rating_average": get_number(data.get("rating_average")),
            "review_count": get_number(data.get("review_count")),
            "quantity_sold": sold,
            "all_time_quantity_sold": get_number(
                data.get("all_time_quantity_sold")
            ),
            "favourite_count": get_number(data.get("favourite_count")),
            "inventory_status": data.get("inventory_status"),
            "stock_qty": get_number(stock_item.get("qty"))
        })

    return rows


# =========================
# 9. ĐỌC FILE ĐẦU VÀO
# =========================

input_df = pd.read_csv(
    INPUT_FILE,
    dtype=str,
    keep_default_na=False
)

required_cols = ["product_id", "seller_product_id"]

missing_cols = [
    col for col in required_cols
    if col not in input_df.columns
]

if missing_cols:
    raise ValueError(
        f"File {INPUT_FILE} thiếu cột: {missing_cols}"
    )

if START_ROW < 1:
    raise ValueError("START_ROW phải từ 1 trở lên.")

if END_ROW is not None and END_ROW < START_ROW:
    raise ValueError("END_ROW phải >= START_ROW.")

# START_ROW và END_ROW đều tính theo dòng dữ liệu, không tính header
if END_ROW is None:
    selected_df = input_df.iloc[START_ROW - 1:]
else:
    selected_df = input_df.iloc[START_ROW - 1:END_ROW]

selected_df = selected_df.reset_index(drop=True)

print(f"Tổng số dòng cần xử lý: {len(selected_df)}")


# =========================
# 10. GỌI API VÀ THU THẬP
# =========================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*"
})

rows = {
    "product": [],
    "category": [],
    "seller": [],
    "history": []
}

processed = 0
failed = 0
access_denied = False

for index, row in selected_df.iterrows():
    product_id = row["product_id"].strip()
    seller_product_id = row["seller_product_id"].strip()

    if not product_id or not seller_product_id:
        print(f"Bỏ qua dòng {START_ROW + index}: thiếu ID")
        failed += 1
        continue

    url = (
        f"https://tiki.vn/api/v2/products/{product_id}"
        f"?platform=web&spid={seller_product_id}&version=3"
    )

    print(
        f"[{index + 1}/{len(selected_df)}] "
        f"Product ID: {product_id}, "
        f"Seller Product ID: {seller_product_id}"
    )

    try:
        response = session.get(url, timeout=30)

        # Không tìm cách vượt qua yêu cầu xác thực
        if response.status_code in (401, 403):
            print(
                f"API từ chối truy cập HTTP {response.status_code}. "
                "Dừng crawl; hãy kiểm tra quyền truy cập hợp lệ."
            )
            access_denied = True
            break

        response.raise_for_status()

        try:
            data = response.json()
        except (ValueError, json.JSONDecodeError):
            print("Phản hồi không phải JSON hợp lệ.")
            failed += 1
            continue

        # Một số API có thể trả lỗi trong JSON
        if not isinstance(data, dict) or data.get("id") is None:
            print("JSON không có cấu trúc sản phẩm hợp lệ.")
            failed += 1
            continue

        # Kiểm tra ID phản hồi có khớp sản phẩm yêu cầu không
        if str(data.get("id")) != product_id:
            print("ID trong phản hồi không khớp ID yêu cầu.")
            failed += 1
            continue

        now = datetime.now()
        crawl_time = now.isoformat(timespec="seconds")
        crawl_date = now.date()

        # Bảng 1: product
        rows["product"].append(
            parse_product(data, crawl_date)
        )

        # Bảng 2: product_category
        rows["category"].extend(
            parse_categories(data, data.get("id"))
        )

        # Bảng 3: product_seller
        seller_rows = parse_sellers(data, seller_product_id)
        rows["seller"].extend(seller_rows)

        # Bảng 4: product_price_history
        rows["history"].extend(
            parse_history(
                data, seller_rows, crawl_time, crawl_date
            )
        )

        processed += 1

    except requests.exceptions.HTTPError as e:
        print(f"Lỗi HTTP: {e}")
        failed += 1

    except requests.exceptions.RequestException as e:
        print(f"Lỗi kết nối: {e}")
        failed += 1

    except Exception as e:
        print(f"Lỗi xử lý sản phẩm: {e}")
        failed += 1

    finally:
        # Lưu theo số sản phẩm đã xử lý hoặc lỗi
        if (index + 1) % SAVE_EVERY == 0:
            save_all(rows)
            rows = {name: [] for name in COLUMNS}
            print("Đã lưu dữ liệu trung gian.")

        if not access_denied:
            time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

# Lưu các bản ghi còn lại
save_all(rows)

print("\n========== KẾT QUẢ ==========")
print(f"Thu thập thành công: {processed}")
print(f"Sản phẩm lỗi/bị bỏ qua: {failed}")
print(f"Bị từ chối truy cập: {access_denied}")
print(f"Thư mục kết quả: {OUTPUT_DIR.resolve()}")

for name, path in FILES.items():
    if path.exists():
        print(f"{path.name}: {path.stat().st_size:,} bytes")